In [1]:
import numpy as np
import xspec
import matplotlib.pyplot as plt
from pathlib import Path
import os
from typing import Optional
from datetime import datetime

# Import XSPEC
try:
    from xspec import Xset, AllData, AllModels, Plot, Fit
    XSPEC_AVAILABLE = True
except ImportError:
    XSPEC_AVAILABLE = False
    print("Warning: PyXspec not available")

# Import plotting settings
from xmm_py_spec import plotting_settings
plotting_settings.set_mpl()

# Context manager for changing directory
class ChangeDir:
    """Context manager for changing directory temporarily."""
    def __init__(self, new_path):
        self.new_path = os.path.expanduser(new_path)

    def __enter__(self):
        self.saved_path = os.getcwd()
        os.chdir(self.new_path)

    def __exit__(self, *_):
        os.chdir(self.saved_path)

In [2]:
def plot_xspec_residuals(
    xcm_file: str,
    rebin: int = 20,
    output_dir: Optional[str] = None,
    show_spectrum: bool = False
):
    """
    Plot residuals for all three spectra (PN, M1, M2) from an XSPEC session.
    
    Parameters
    ----------
    xcm_file : str
        Path to .xcm file
    rebin : int
        Rebinning factor for plotting
    output_dir : str, optional
        Directory to save plots. If None, creates plots/ subdirectory next to xcm_file
    show_spectrum : bool, optional
        If True, show spectrum panel in addition to residuals. If False, show only residuals.
        Default is True.
    """
    if not XSPEC_AVAILABLE:
        print("Error: PyXspec not available")
        return
    
    xcm_path = Path(xcm_file)
    if not xcm_path.exists():
        print(f"Error: File not found: {xcm_file}")
        return
    
    # Set output directory
    if output_dir is None:
        output_dir = xcm_path.parent / "plots"
    else:
        output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        # Change to the directory containing the xcm file for relative paths
        with ChangeDir(xcm_path.parent):
            # Clear and restore XSPEC session
            AllData.clear()
            AllModels.clear()
            Plot.device = '/null'
            # Load relxill model library before restoring session
            AllModels.lmod("relxill")
            Xset.restore(xcm_path.name)
            Plot.xAxis = 'keV'
            
            # Extract fit statistics
            try:
                cstat = Fit.statistic
                dof = Fit.dof
                cstat_reduced = cstat / dof if dof > 0 else None
            except Exception as e:
                print(f"Warning: Could not extract fit statistics: {e}")
                cstat = None
                dof = None
                cstat_reduced = None
            
            # Set rebinning
            Plot.setRebin(rebin, rebin)
            Plot("eeufs ra")
            
            # Define groups and their labels/colors
            groups = [
                (1, "PN", "C0"),
                (2, "M1", "C1"),
                (3, "M2", "C2")
            ]
            
            # Create figure with one or two subplots
            if show_spectrum:
                fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), 
                                              gridspec_kw={'height_ratios': [2, 1]})
            else:
                fig, ax1 = plt.subplots(1, 1, figsize=(10, 5))
                ax2 = None
            
            # Extract and plot data for each group
            for group_num, label, color in groups:
                try:
                    # Extract plot data
                    energies = Plot.x(group_num, 1)
                    edeltas = Plot.xErr(group_num, 1)
                    rates = Plot.y(group_num, 1)
                    errors = Plot.yErr(group_num, 1)
                    foldedmodel = Plot.model(group_num)
                    
                    # Prepare step energies for model plot
                    nE = len(energies)
                    stepenergies = []
                    for i in range(nE):
                        stepenergies.append(energies[i] - edeltas[i])
                    stepenergies.append(energies[-1] + edeltas[-1])
                    foldedmodel.append(foldedmodel[-1])
                    
                    # Get residuals
                    resid = Plot.y(group_num, 2)
                    residerr = Plot.yErr(group_num, 2)

                    # Plot residuals on ax1
                    ax1.set_xscale('log')
                    ax1.errorbar(
                        energies, resid, xerr=edeltas, yerr=residerr,
                        fmt='.', color=color, alpha=0.8, label=label
                    )
                    
                    # Plot spectrum on ax2 (if enabled)
                    if ax2 is not None:
                        ax2.set_xscale('log')
                        ax2.set_yscale('log')
                        ax2.errorbar(
                            energies, rates, xerr=edeltas, yerr=errors,
                            fmt='.', alpha=0.8, color=color, zorder=10, label=label
                        )
                        ax2.scatter(
                            energies, rates, color=color, s=1, zorder=10
                        )
                        ax2.plot(
                            stepenergies, foldedmodel, color=color, lw=1,
                            alpha=0.6, linestyle='--'
                        )
                    
                    
                except Exception as e:
                    print(f"Warning: Failed to plot group {group_num} ({label}): {e}")

            # Format residuals panel
            title_text = xcm_path.stem
            if cstat is not None and dof is not None and cstat_reduced is not None:
                title_text += f"\nC-stat = {cstat:.1f}, DOF = {dof}, C-stat/DOF = {cstat_reduced:.3f}"
            ax1.set_title(title_text, fontsize=15, weight='bold', y=0.99)
            ax1.axhline(1, ls='-', color='gray', alpha=0.4, linewidth=1)
            ax1.set_ylim(-1, 3)
            ax1.set_xlim(1, 11)
            ax1.set_xlabel("Energy (keV)", fontsize=12)
            ax1.set_ylabel("data / model", fontsize=12)
            ax1.legend(loc='best', fontsize=10)
            ax1.tick_params(labelsize=10)
            ax1.grid(True, alpha=0.3)
            
            # Format spectrum panel (if enabled)
            if ax2 is not None:
                ax2.set_xlabel(None)
                ax2.set_xlim(1, 11)
                ax2.set_ylim(1e-9, 1e-3)
                ax2.set_ylabel("Rate (counts s$^{-1}$ keV$^{-1}$)", fontsize=12)
                ax2.legend(loc='best', fontsize=10)
                ax2.tick_params(labelsize=10)
                ax2.grid(True, alpha=0.3)
            
            # Save figure with timestamp
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            output_filename = output_dir / f"{xcm_path.stem}_residuals_{timestamp}.png"
            plt.tight_layout()
            plt.savefig(output_filename, dpi=300, bbox_inches='tight')
            plt.close()
            
            print(f"Plot saved to: {output_filename}")
            
    except Exception as e:
        print(f"Error loading {xcm_file}: {e}")
        import traceback
        traceback.print_exc()


In [ ]:
# Define path to XSPEC sessions
# Get workspace root (assuming notebook is in notebooks/ subdirectory)
workspace_root = Path(__file__).parent.parent if '__file__' in globals() else Path.cwd().parent
# If running in notebook, try to find the workspace root
try:
    # Try to import from the package - if it works, we can infer the workspace root
    import xmm_py_spec
    workspace_root = Path(xmm_py_spec.__file__).parent.parent.parent
except:
    # Fallback: assume we're in the workspace root or one level up
    if Path("data").exists():
        workspace_root = Path.cwd()
    elif Path("../data").exists():
        workspace_root = Path.cwd().parent
    else:
        # Try absolute path
        workspace_root = Path("/home/mike/Repos/iki/xmm_py_spec")

base_dir = workspace_root / "data/downloaded_spectra/201237001010017_5359_old/combined/PN_M1_M2"

print(f"Using workspace root: {workspace_root}")
print(f"Looking for files in: {base_dir}")
print(f"Directory exists: {base_dir.exists()}")

# Plot residuals for both XSPEC sessions
xcm_files = [
    base_dir / "mo2_relxill_240625_no_broken_po.xcm",
    base_dir / "Absorbed power law with relxill.xcm",
    base_dir / "Absorbed power law.xcm",
]

for xcm_file in xcm_files:
    print(f"\nProcessing: {xcm_file.name}")
    print(f"File exists: {xcm_file.exists()}")
    if xcm_file.exists():
        plot_xspec_residuals(str(xcm_file), rebin=20)
    else:
        print(f"  Skipping: File not found at {xcm_file}")


Using workspace root: /Users/mike/Repos/iki/xmm_py_spec
Looking for files in: /Users/mike/Repos/iki/xmm_py_spec/data/downloaded_spectra/201237001010017_5359_old/combined/PN_M1_M2
Directory exists: True

Processing: mo2_relxill_240625_no_broken_po.xcm
File exists: True
Error loading /Users/mike/Repos/iki/xmm_py_spec/data/downloaded_spectra/201237001010017_5359_old/combined/PN_M1_M2/mo2_relxill_240625_no_broken_po.xcm: Error attempting to load local model library.

Processing: Absorbed power law with relxill.xcm
File exists: True
Error loading /Users/mike/Repos/iki/xmm_py_spec/data/downloaded_spectra/201237001010017_5359_old/combined/PN_M1_M2/Absorbed power law with relxill.xcm: Error attempting to load local model library.

Processing: Absorbed power law.xcm
File exists: True
Error loading /Users/mike/Repos/iki/xmm_py_spec/data/downloaded_spectra/201237001010017_5359_old/combined/PN_M1_M2/Absorbed power law.xcm: Error attempting to load local model library.


 No valid sTraceback (most recent call last):
  File "/var/folders/7_/h2xhyhcd4c39tz62r5nh23r40000gn/T/ipykernel_46655/2847531386.py", line 46, in plot_xspec_residuals
    AllModels.lmod("relxill")
  File "/Users/mike/Soft/heasoft-6.31.1/aarch64-apple-darwin22.5.0/lib/python/xspec/model.py", line 1100, in lmod
    _pyXspec.localModel(packageName, dPath)
Exception: Error attempting to load local model library.
etting for the default local model  directory. 
Set a path for the model library or  modify the default setting in Xspec.init
 No valid setting for the default local model  directory. 
Set a path for the model library or  modify the default setting in Xspec.init
Traceback (most recent call last):
  File "/var/folders/7_/h2xhyhcd4c39tz62r5nh23r40000gn/T/ipykernel_46655/2847531386.py", line 46, in plot_xspec_residuals
    AllModels.lmod("relxill")
  File "/Users/mike/Soft/heasoft-6.31.1/aarch64-apple-darwin22.5.0/lib/python/xspec/model.py", line 1100, in lmod
    _pyXspec.localModel